<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/06_multi_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 06 — Multi-Agent Hierarchies

> **Where you are** — the previous course's function-calling video promised "sub-agents callable as functions." This module is that sentence, made precise.
> - **You can already:** wrap an agent as a tool (`AgentTool` — the translator specialist from M02).
> - **New in this module:** the second pattern — `sub_agents` with LLM-driven transfer — and how to choose between the two.
> - **No new Python.**

Back to the IT help desk for a moment. Some requests deserve a **hand-off**: *"my laptop won't boot"* starts a long back-and-forth that a hardware specialist should own. Other requests deserve a **quick consultation**: *"translate this reply"* is one question, one answer, and the desk keeps talking to the employee itself.

ADK has one pattern for each:

- **`sub_agents`** — the coordinator *transfers* the conversation to a specialist, who takes over. Like handing the phone to a colleague.
- **`AgentTool`** — the coordinator *asks* a specialist and writes the reply itself. You know this one — the translator from M02.

**What we'll do:** build the same coordinator-plus-two-specialists team both ways, watch how differently the event streams look, and close with an honest question: when is multi-agent worth it at all?

**Running cost:** under $0.01.

# Setup

The usual ritual — install, key, imports. The odd-looking lines in the imports are the same Jupyter plumbing as before; the comments explain them.

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


## API Key

In [2]:
import os
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
MODEL_STRING = "openrouter/openai/gpt-5.6-luna"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/openai/gpt-5.6-luna


## Imports

In [3]:
import os
import sys, warnings, asyncio, uuid, logging
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")
import nest_asyncio; nest_asyncio.apply()
os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)
logging.getLogger("google_adk").setLevel(logging.ERROR)  # hide ADK 2.7 context-cache advisory on transfers

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.agent_tool import AgentTool
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


# Three Ways to Combine Agents

You now have three ways to put agents together:

| Pattern | Who decides what runs | When |
|---|---|---|
| **Workflow agent** (M05) | You do — Sequential, Parallel, Loop | You can name the steps in advance |
| **`sub_agents`** (this module) | The LLM — by transferring | The specialist should own the dialog |
| **`AgentTool`** (this module) | The LLM — by calling a function | The coordinator stays in charge |

The two new patterns both let the LLM decide what runs next. The whole difference — and honestly the whole module — is one question: **who is in charge after the decision?**

# Pattern 1 — `sub_agents`: Hand Over the Conversation

*"My laptop won't boot."* That opens a diagnosis conversation the hardware specialist should own. The key line in the cell below:

```python
sub_agents=[greeter, weather_specialist]
```

That one parameter changes the coordinator's job. Any agent with `sub_agents=` automatically gets a built-in tool called `transfer_to_agent(agent_name=...)` — you don't register it, ADK injects it. When the coordinator's LLM decides a specialist fits better, it calls that tool, and ADK **hands the conversation over**: the specialist runs, and *the specialist's answer is what the user sees*. Handing the phone to a colleague.

One thing to write with care: the child's `description=`. It is the only thing the coordinator's LLM reads when deciding where to route — the same job the docstring does for a tool.

In [4]:
# Two specialists — greeter and weather_specialist.
greeter = LlmAgent(
    name="greeter",
    model=LiteLlm(model=MODEL_STRING),
    description="Handles greetings and casual chat.",
    instruction="You are a friendly greeter. Respond warmly in one short sentence.",
)

weather_specialist = LlmAgent(
    name="weather_specialist",
    model=LiteLlm(model=MODEL_STRING),
    description="Handles weather questions for any city.",
    instruction=(
        "You are a weather assistant. You do not have real weather tools; "
        "produce plausible one-sentence weather reports for the city asked about."
    ),
)

# The coordinator routes via sub_agents.
coordinator_subagents = LlmAgent(
    name="coordinator_subagents",
    model=LiteLlm(model=MODEL_STRING),
    description="Coordinator that routes to specialists via transfer.",
    instruction="""You coordinate a small team. You have two specialists available:
- `greeter` for greetings and casual chat.
- `weather_specialist` for weather questions.

When the user's message fits one of them, use transfer_to_agent to hand over.
Do not answer greetings or weather questions yourself — always delegate.""",
    sub_agents=[greeter, weather_specialist],
)

print("✅ Coordinator (sub_agents pattern) ready.")

✅ Coordinator (sub_agents pattern) ready.


The `chat()` helper again — same as always, except it prints each event's **author**, because in this module the author is the whole story.

In [5]:
APP = "m06_demos"
USER = "student"
session_service = InMemorySessionService()

async def chat(agent, prompt: str, session_id: str = None):
    sid = session_id or f"s-{uuid.uuid4().hex[:6]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"USER: {prompt}\n")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text and p.text.strip():
                    print(f"[{ev.author}] {p.text.strip()[:200]}")
                if p.function_call:
                    args = dict(p.function_call.args) if p.function_call.args else {}
                    print(f"[{ev.author}] → {p.function_call.name}({args})")
                if p.function_response:
                    resp = str(p.function_response.response)
                    if len(resp) < 200:
                        print(f"[tool_resp] {resp}")

print("✅ chat() ready.")

✅ chat() ready.


## Run It — Watch Who Answers

Two questions — one casual, one about weather. In each event stream, look at who signs the final answer.

In [6]:
await chat(coordinator_subagents, "Hi there, how are you?")

USER: Hi there, how are you?



[coordinator_subagents] → transfer_to_agent({'agent_name': 'greeter'})
[tool_resp] {'result': None}


[greeter] Hi there! I’m doing well, thanks for asking—how are you?


In [7]:
await chat(coordinator_subagents, "What's the weather in Prague?")

USER: What's the weather in Prague?



[coordinator_subagents] → transfer_to_agent({'agent_name': 'weather_specialist'})
[tool_resp] {'result': None}


[weather_specialist] **Considering Prague weather**

I need to provide a plausible one-sentence answer about the weather in Prague. I think it’s acceptable to avoid mentioning that it’s live data, maybe focusing on a gene
[weather_specialist] Prague is experiencing partly cloudy skies with mild temperatures around 15°C (59°F) and a light breeze.


### 🔍 What just happened?

- The coordinator answered neither question itself. It emitted `transfer_to_agent(agent_name='...')` — a tool call — and ADK routed.
- Look at the authors: the final answers came from `[greeter]` and `[weather_specialist]`, not from the coordinator. **The specialist owns the reply.** That is the transfer.

One honest detail for real projects: after a transfer, the specialist *stays* the active agent for that session. A follow-up question goes to the specialist, not back to the coordinator, unless someone transfers again. (Which agent is active is stored in the session, next to the state you met in M03.)

### 🎯 Mini-task

Add a third specialist, `joke_specialist`, to `sub_agents=[...]` and ask for a joke — does the routing hold? Then run two `chat(...)` calls with the same `session_id="test-1"` — a weather question, then a follow-up — and check who answers the second one.

# Pattern 2 — `AgentTool`: Ask a Specialist, Stay in Charge

Same team, different wiring — and you have seen this move before: it is exactly how the help desk consulted the translator in M02. The key line:

```python
tools=[AgentTool(agent=greeter_t), AgentTool(agent=weather_specialist_t)]
```

No `sub_agents=`, no transfer. Each specialist is wrapped as a **tool**, so to the coordinator's LLM it looks just like `get_current_weather` did: a name, a description, something to call. When the coordinator uses one, ADK runs the specialist, brings its answer back as a tool response, and **the coordinator writes the final reply itself**. The specialist answers one question and steps back — it never talks to the user directly.

In [8]:
# Same specialists; same descriptions; different wiring.
# Use separate instances so they don't accidentally share parent state.
greeter_t = LlmAgent(
    name="greeter",
    model=LiteLlm(model=MODEL_STRING),
    description="Handles greetings and casual chat. Input: user message. Output: one warm sentence.",
    instruction="Respond warmly in one short sentence.",
)

weather_specialist_t = LlmAgent(
    name="weather_specialist",
    model=LiteLlm(model=MODEL_STRING),
    description="Reports weather for any city. Input: the city name. Output: one-sentence weather report.",
    instruction="Produce a plausible one-sentence weather report.",
)

coordinator_tools = LlmAgent(
    name="coordinator_tools",
    model=LiteLlm(model=MODEL_STRING),
    description="Coordinator that calls specialists as tools.",
    instruction="""You have exactly two tools available, and you must use one of them
for every user message:

- Call the tool named `greeter` for greetings, small talk, or casual chat.
- Call the tool named `weather_specialist` for any weather question.

Do NOT invent other tool names. Do NOT answer directly. Always call one of
those two tools, pass it the user's message as the `request` argument, read
the result, and wrap it in your own friendly reply.""",
    tools=[
        AgentTool(agent=greeter_t),
        AgentTool(agent=weather_specialist_t),
    ],
)

print("✅ Coordinator (AgentTool pattern) ready.")

✅ Coordinator (AgentTool pattern) ready.


## Run It — Same Questions, Different Signature

The same two questions. This time, watch who signs the final answer.

In [9]:
await chat(coordinator_tools, "Hi there, how are you?")

USER: Hi there, how are you?



[coordinator_tools] → greeter({'request': 'Hi there, how are you?'})


[tool_resp] {'result': 'Hi there! I’m doing well, thanks for asking—how are you?'}


[coordinator_tools] Hi there! I’m doing well, thanks for asking—how are you?


In [10]:
await chat(coordinator_tools, "What's the weather in Prague?")

USER: What's the weather in Prague?



[coordinator_tools] → weather_specialist({'request': "What's the weather in Prague?"})


[tool_resp] {'result': 'Prague is experiencing partly cloudy skies with mild temperatures and a light breeze.'}


[coordinator_tools] Prague is experiencing partly cloudy skies with mild temperatures and a light breeze.


### 🔍 What just happened?

- The coordinator emitted a regular tool call — `weather_specialist(...)` — not `transfer_to_agent`.
- A `[tool_resp]` event carried the specialist's answer back.
- The final answer's author is `[coordinator_tools]`. The coordinator read the specialist's output and wrote its own reply — the specialist never had the mic.

### 🎯 Mini-task

Ask the consultant coordinator: *"Greet me, and tell me the weather in Warsaw."* Does it call both specialists in one turn? In which order do the tool calls appear?

# Side by Side

Same input, same team, two wirings. The event streams tell you which pattern you are looking at.

## `sub_agents` — the transfer

```
USER: What's the weather in Prague?

[coordinator_subagents] → transfer_to_agent(agent_name='weather_specialist')
[weather_specialist]    Prague is cloudy and cool, with a gentle breeze.
                        ^^^ final answer comes from the specialist
```

## `AgentTool` — the consultation

```
USER: What's the weather in Prague?

[coordinator_tools] → weather_specialist(request='...')
[tool_resp]            {'result': 'In Prague, expect a partly cloudy day...'}
[coordinator_tools]    The weather in Prague is partly cloudy with a gentle breeze.
                       ^^^ final answer comes from the coordinator,
                           wrapping the specialist's output
```

**Quick check for any event stream:** who authored the final answer? A specialist → transfer. The coordinator → consultation.

# Which Pattern When?

| Use `sub_agents` (transfer) when... | Use `AgentTool` (consultant) when... |
|---|---|
| The specialist should **own the conversation** after routing | The specialist should **answer one question and step back** |
| The child might take several turns with the user | The child has a clean input/output contract |
| The topic shifted — the specialist is the right conversation partner now | The coordinator combines the specialist's output with other sources |
| The user should feel they are talking to a specialist | The user should feel they talk to one assistant with specialists behind it |

A practical rule: if the specialist's work is part of a **larger answer**, use `AgentTool`. If the specialist's work *is* the answer, use `sub_agents`.

One concrete scenario for each:

**`sub_agents`.** An IT support desk with a billing specialist and a hardware specialist. The user says *"my laptop won't boot"*. The coordinator routes to hardware, and over the next five turns the hardware specialist walks the user through diagnosis. An extended conversation the coordinator shouldn't mediate — transfer is right.

**`AgentTool`.** A coding assistant whose orchestrator wraps a code-analyzer, a docs-lookup, and a test-runner. The user asks *"why is my test failing?"* The orchestrator calls all three, reads their outputs, and writes one combined answer. Each specialist contributes a single structured response — consultation is right.

# When NOT to Go Multi-Agent

> *Two minutes of honesty before you architect.*

Multi-agent architectures are fashionable. They are also expensive. Every specialist you add brings a **coordination cost**, and it has three parts:

1. **Extra LLM calls.** Every transfer or specialist call is one more model call. A two-specialist system makes at least two calls per turn; a five-specialist system can make ten. Latency and cost add up.
2. **Routing errors.** Every routing decision is a chance to pick the wrong specialist. More specialists, more ways to miss.
3. **Fragmented context.** Each child has its own system prompt. What you told the coordinator does not automatically reach the children — specialists sometimes fail simply because they lack context the coordinator had.

## Three Tests Before You Split

- **Reuse.** Will a specialist also serve elsewhere — another top-level agent, another product? If yes, splitting creates reusable pieces. If no, more tools in one agent may do.
- **Different models.** Would each specialist benefit from a different model — cheap routing, expensive answering? Separate agents let you mix models per task. Same model everywhere? The split buys less.
- **Instruction size.** Is one system prompt becoming unmanageable — hundreds of lines of overlapping rules? Splitting gives each agent a short, focused prompt. A 30-line prompt does not need splitting.

If none of the three passes, **keep one agent with tools**. A single well-written agent with eight tools is almost always simpler, cheaper and faster than a coordinator-plus-specialists doing the same work. Multi-agent is a deliberate choice, not a default.

### 🎯 Mini-task

Take the guarded ticket agent from M02 and run it through the three tests. Would splitting it into several agents be justified? Two or three sentences per test.

# Key Takeaways

- **`sub_agents` = hand over the conversation.** The coordinator's LLM calls the built-in `transfer_to_agent(...)`; the specialist owns the reply — and stays active for follow-ups in the session.
- **`AgentTool` = ask and stay in charge.** The specialist is called like a function; the coordinator writes the final reply.
- **The child's `description=` is the routing schema.** The coordinator's LLM decides from it alone — write it for the model, like a docstring.
- **Quick check in any event stream:** who authored the final answer? Specialist → transfer. Coordinator → consultation.
- **Multi-agent has a coordination cost** — extra calls, routing misses, fragmented context. Split only when the reuse, different-models, or instruction-size test passes. Default: one agent with tools.

# Next up — M07: Callbacks

Six places where your own function can run in the middle of an agent's work — before and after the model, before and after every tool, before and after the whole agent. Return nothing and the run continues; return a value and you just replaced that step. The safety guardrail demo is the highlight. See you there.